# 1.04 — Presentation Analytics

**Author:** Yashaswi Mohanty  
**Date:** 2026-03-28  
**Purpose:** Produce the three pending calculations for the Monday presentation.

1. **Petitioner frequency distribution table** — how many petitioners file 1, 2, 3, … complaints (excl. discarded)
2. **Repeat-filer subcategory analysis** — for petitioners who file ≥2 complaints, do they re-file the same subcategory (resolution failure) or different subcategories (active citizen)?
3. **Disposed-only resolution time histogram** — distribution of disposal days for `status = 'Disposed'` across all four fiscal years
4. **Rural housing regression** — OLS of disposal time on district FE vs block FE, restricted to Rural Housing + PMAY subcategories

All analysis uses complaints excluding `status = 'Discard'` unless otherwise noted.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path('__file__').resolve().parent.parent
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import warnings
warnings.filterwarnings('ignore')

import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.formula.api as smf
from IPython.display import display

from app.config import load_duckdb, directories

# ── DPIC / UChicago style ────────────────────────────────────────────────────
MAROON       = '#800000'   # UChicago maroon — primary chart colour
MAROON_MID   = '#A84040'   # mid-tone for secondary elements
MAROON_LIGHT = '#D4A0A0'   # light for annotations / secondary series
NEAR_BLACK   = '#1A1A1A'   # primary text
GREY_AXIS    = '#AAAAAA'   # spine / tick colour
GREY_GRID    = '#E8E8E8'   # grid line colour

# Four-year palette: light → dark maroon (oldest → most recent)
FY_COLORS = ['#D4A0A0', '#B05050', '#8C2E2E', MAROON]

plt.rcParams.update({
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'axes.edgecolor':      GREY_AXIS,
    'axes.grid':           True,
    'axes.grid.axis':      'y',
    'grid.color':          GREY_GRID,
    'grid.linewidth':      0.6,
    'font.family':         'sans-serif',
    'font.size':           10,
    'axes.titlesize':      11,
    'axes.titleweight':    'bold',
    'axes.titlecolor':     NEAR_BLACK,
    'axes.labelcolor':     NEAR_BLACK,
    'xtick.color':         '#555555',
    'ytick.color':         '#555555',
    'xtick.labelsize':     9,
    'ytick.labelsize':     9,
})

OUTPUT = directories.OUTPUT
print(f'Output dir: {OUTPUT}')


## 0. Load and prepare data

In [ ]:
# Try duckdb loader first (works in full uv env); fall back to sqlite3 if
# duckdb extension download is unavailable (e.g. air-gapped / no network).
try:
    df_raw = load_duckdb(output_format="polars")
    print("Loaded via DuckDB")
except Exception as e:
    print(f"DuckDB unavailable ({e}), falling back to sqlite3")
    import sqlite3
    _conn = sqlite3.connect(str(directories.RAW_DATA / "grievance.db"))
    df_raw = pl.read_database("SELECT * FROM complaints", _conn)
    _conn.close()

print(f"Raw rows: {len(df_raw):,}")
print(f"Columns: {df_raw.columns}")

In [ ]:
# ── Cast dates (no-op if already Datetime, handles string fallback) ───────────
# duckdb sqlite_scanner infers DATETIME columns correctly; direct sqlite3 load
# returns them as strings. cast() covers both.

df = (
    df_raw
    .with_columns([
        pl.col("created_on").cast(pl.Datetime, strict=False),
        pl.col("resolved_on").cast(pl.Datetime, strict=False),
        (pl.col("petitioner_name") + "_" + pl.col("petitioner_mobile")).alias("petitioner_id"),
    ])
    .with_columns([
        pl.col("created_on").dt.year().alias("year"),
        pl.col("created_on").dt.month().alias("month"),
        pl.when(pl.col("created_on").dt.month() >= 4)
          .then(pl.col("created_on").dt.year())
          .otherwise(pl.col("created_on").dt.year() - 1)
          .alias("fy_start"),
    ])
    .with_columns(
        (pl.col("fy_start").cast(pl.Utf8) + "-" +
         (pl.col("fy_start") + 1).cast(pl.Utf8).str.slice(2, 2)).alias("fiscal_year")
    )
)

print(df.group_by("status").len().sort("len", descending=True))
print(f"\nFiscal years in data:")
print(df.group_by("fiscal_year").len().sort("fiscal_year"))

In [ ]:
# ── Working dataset: exclude discarded ───────────────────────────────────────
DISCARD_STATUS = "Discard"
RURAL_HOUSING_SUBCATS = ["Rural Housing", "IAY/MKY/BPGY/PMAY"]

df_clean = df.filter(pl.col("status") != DISCARD_STATUS)
print(f"After excluding discards: {len(df_clean):,} rows (dropped {len(df_raw)-len(df_clean):,})")

# Focus FYs for maps / primary analysis
PRIMARY_FYS = ["2021-22", "2022-23", "2023-24", "2024-25"]

---
## 1. Petitioner frequency distribution table

For each petitioner (identified by `name_mobile`), count how many complaints they filed (excl. discarded).  
Output: table of `n_complaints → n_petitioners → share_of_petitioners → cumulative_share`.

In [ ]:
# ── Complaints per petitioner ─────────────────────────────────────────────────
pet_counts = (
    df_clean
    .group_by("petitioner_id")
    .agg(pl.len().alias("n_complaints"))
)

total_petitioners = len(pet_counts)
print(f"Unique petitioners (excl. discarded): {total_petitioners:,}")
print(f"Summary stats:")
print(pet_counts.select("n_complaints").describe())

In [ ]:
# ── Frequency distribution table ─────────────────────────────────────────────
MAX_EXPLICIT = 10  # show rows 1-10 individually; group 11+ together

freq = (
    pet_counts
    .with_columns(
        pl.when(pl.col("n_complaints") <= MAX_EXPLICIT)
          .then(pl.col("n_complaints").cast(pl.Utf8))
          .otherwise(pl.lit(f"{MAX_EXPLICIT+1}+"))
          .alias("complaints_bucket")
    )
    .group_by("complaints_bucket")
    .agg(pl.len().alias("n_petitioners"))
    .with_columns([
        (pl.col("n_petitioners") / total_petitioners * 100).round(2).alias("share_pct"),
    ])
    .sort("complaints_bucket",
          descending=False,
          nulls_last=True)
)

# Custom sort: numeric rows first, then '11+'
freq_pd = freq.to_pandas()
def bucket_sort_key(b):
    try: return int(b)
    except: return 9999
freq_pd["_sort"] = freq_pd["complaints_bucket"].apply(bucket_sort_key)
freq_pd = freq_pd.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

# Cumulative share
freq_pd["cumulative_share_pct"] = freq_pd["share_pct"].cumsum().round(2)

freq_pd.columns = ["No. of complaints", "No. of petitioners", "Share (%)", "Cumulative share (%)"]
display(freq_pd)

In [ ]:
# ── Save frequency table to Excel ────────────────────────────────────────────
out_path = OUTPUT / "tables" / "petitioner_frequency_distribution.xlsx"
out_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    freq_pd.to_excel(writer, sheet_name="frequency_table", index=False)
    
    # Summary stats sheet
    stats = pet_counts.select("n_complaints").describe().to_pandas()
    stats.to_excel(writer, sheet_name="summary_stats", index=False)

print(f"Saved: {out_path}")

---
## 2. Repeat-filer subcategory analysis

For petitioners who filed ≥2 complaints (excl. discarded): do they re-file the **same subcategory** (resolution failure / persistence) or **different subcategories** (active citizen with multiple distinct grievances)?

In [ ]:
# ── Repeat filers only ────────────────────────────────────────────────────────
repeat_filers = pet_counts.filter(pl.col("n_complaints") >= 2).select("petitioner_id")
print(f"Petitioners filing ≥2 complaints: {len(repeat_filers):,} ({len(repeat_filers)/total_petitioners*100:.1f}% of all petitioners)")

df_repeat = df_clean.join(repeat_filers, on="petitioner_id", how="inner")
print(f"Complaints from repeat filers: {len(df_repeat):,}")

In [ ]:
# ── Distinct subcategories per repeat-filing petitioner ───────────────────────
subcat_diversity = (
    df_repeat
    .group_by("petitioner_id")
    .agg([
        pl.len().alias("n_complaints"),
        pl.col("subcategory").n_unique().alias("n_distinct_subcategories"),
        pl.col("subcategory").first().alias("first_subcategory"),
    ])
    .with_columns(
        pl.when(pl.col("n_distinct_subcategories") == 1)
          .then(pl.lit("Same subcategory (re-filing)"))
          .otherwise(pl.lit("Different subcategories"))
          .alias("filing_pattern")
    )
)

pattern_summary = (
    subcat_diversity
    .group_by("filing_pattern")
    .agg(pl.len().alias("n_petitioners"))
    .with_columns(
        (pl.col("n_petitioners") / pl.col("n_petitioners").sum() * 100).round(1).alias("share_pct")
    )
    .sort("n_petitioners", descending=True)
)

print("Filing pattern among repeat filers:")
display(pattern_summary.to_pandas())

In [ ]:
# ── Among same-subcategory re-filers: what subcategory? ───────────────────────
same_subcat = subcat_diversity.filter(pl.col("filing_pattern") == "Same subcategory (re-filing)")

top_resubmit_subcats = (
    same_subcat
    .group_by("first_subcategory")
    .agg(pl.len().alias("n_petitioners"))
    .sort("n_petitioners", descending=True)
    .head(15)
)

print("Top subcategories among same-subcategory re-filers:")
display(top_resubmit_subcats.to_pandas())

In [ ]:
# ── By complaints-per-petitioner band: same vs different ─────────────────────
band_pattern = (
    subcat_diversity
    .with_columns(
        pl.when(pl.col("n_complaints") == 2).then(pl.lit("2"))
          .when(pl.col("n_complaints") <= 5).then(pl.lit("3–5"))
          .when(pl.col("n_complaints") <= 10).then(pl.lit("6–10"))
          .otherwise(pl.lit("11+"))
          .alias("complaint_band")
    )
    .group_by(["complaint_band", "filing_pattern"])
    .agg(pl.len().alias("n_petitioners"))
    .sort(["complaint_band", "filing_pattern"])
)

# Pivot for readability
band_pivot = band_pattern.to_pandas().pivot(
    index="complaint_band", columns="filing_pattern", values="n_petitioners"
).fillna(0).astype(int)
band_pivot["total"] = band_pivot.sum(axis=1)
band_pivot["same_pct"] = (band_pivot.get("Same subcategory (re-filing)", 0) / band_pivot["total"] * 100).round(1)

print("Filing pattern by complaint frequency band:")
display(band_pivot)

In [ ]:
# ── Save repeat-filer analysis ────────────────────────────────────────────────
out_path_subcat = OUTPUT / "tables" / "repeat_filer_subcategory_analysis.xlsx"

with pd.ExcelWriter(out_path_subcat, engine="openpyxl") as writer:
    pattern_summary.to_pandas().to_excel(writer, sheet_name="pattern_summary", index=False)
    top_resubmit_subcats.to_pandas().to_excel(writer, sheet_name="top_resubmit_subcats", index=False)
    band_pivot.to_excel(writer, sheet_name="by_complaint_band")

print(f"Saved: {out_path_subcat}")

---
## 3. Disposed-only resolution time histogram

Distribution of disposal time (days from `created_on` to `resolved_on`) for complaints with `status = 'Disposed'`, across all four fiscal years. This is the correct measure of genuine resolution speed — not the headline median which is pulled down by fast-closing discards.

In [ ]:
# ── Disposal time for disposed complaints ─────────────────────────────────────
df_disposed = (
    df
    .filter(pl.col("status") == "Disposed")
    .filter(pl.col("resolved_on").is_not_null())
    .with_columns(
        ((pl.col("resolved_on") - pl.col("created_on")).dt.total_days()).alias("disposal_days")
    )
    .filter(pl.col("disposal_days") >= 0)
    .filter(pl.col("fiscal_year").is_in(PRIMARY_FYS))
)

print(f"Disposed complaints with valid resolution date: {len(df_disposed):,}")
print(df_disposed.group_by("fiscal_year").agg([
    pl.len().alias("n"),
    pl.col("disposal_days").median().alias("median_days"),
    pl.col("disposal_days").mean().alias("mean_days"),
    pl.col("disposal_days").quantile(0.9).alias("p90_days"),
]).sort("fiscal_year"))

In [ ]:
# ── Histogram: disposal time by fiscal year ───────────────────────────────────
CAP_DAYS = 180

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
axes = axes.flatten()

for ax, fy in zip(axes, PRIMARY_FYS):
    data = (
        df_disposed
        .filter(pl.col('fiscal_year') == fy)
        .select('disposal_days')
        .to_pandas()['disposal_days']
        .clip(upper=CAP_DAYS)
    )
    median_d = data.median()
    n = len(data)

    ax.hist(data, bins=60, color=MAROON, alpha=0.88, edgecolor='white', linewidth=0.2)
    ax.axvline(median_d, color=NEAR_BLACK, linewidth=1.4, linestyle='--')

    # Place label left of line when median is near the cap, right otherwise
    near_cap = median_d > CAP_DAYS * 0.8
    x_label  = median_d - 4 if near_cap else median_d + 3
    ha_label = 'right'      if near_cap else 'left'
    ax.text(x_label, ax.get_ylim()[1] * 0.92,
            f'Median: {median_d:.0f}d', fontsize=8, color=NEAR_BLACK, ha=ha_label)

    ax.set_title(f'FY {fy}', fontsize=11)
    ax.set_xlabel('Days to disposal', fontsize=9, color='#555555')
    ax.set_ylabel('No. of complaints', fontsize=9, color='#555555')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.text(0.97, 0.97, f'n = {n:,}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='#777777')
    ax.spines['left'].set_color(GREY_AXIS)
    ax.spines['bottom'].set_color(GREY_AXIS)

fig.suptitle('Distribution of disposal time — disposed complaints only',
             fontsize=13, fontweight='bold', color=NEAR_BLACK, y=1.01)
fig.text(0.5, -0.02,
         f'Capped at {CAP_DAYS} days for display. Dashed line = median.',
         ha='center', fontsize=8, color='#777777', style='italic')

plt.tight_layout()
fig_path = OUTPUT / 'disposal_time_histogram.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()


In [ ]:
# ── Single overlaid histogram for presentation ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for fy, color in zip(PRIMARY_FYS, FY_COLORS):
    data = (
        df_disposed
        .filter(pl.col('fiscal_year') == fy)
        .select('disposal_days')
        .to_pandas()['disposal_days']
        .clip(upper=CAP_DAYS)
    )
    median_d = data.median()
    ax.hist(data, bins=60, alpha=0.60, color=color,
            label=f'FY {fy}  (median {median_d:.0f}d)', density=True)

ax.set_xlabel('Days to disposal', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Resolution time improving year on year (disposed complaints only)', fontsize=12)
ax.legend(fontsize=9, framealpha=0.9, edgecolor=GREY_GRID)
ax.spines['left'].set_color(GREY_AXIS)
ax.spines['bottom'].set_color(GREY_AXIS)
fig.text(0.5, -0.04,
         f'Capped at {CAP_DAYS} days. Density normalised so years are visually comparable.',
         ha='center', fontsize=8, color='#777777', style='italic')

plt.tight_layout()
fig_path2 = OUTPUT / 'disposal_time_overlaid.png'
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path2}')
plt.show()


In [ ]:
# ── Median disposal time bar chart (July-June fiscal years) ──────────────────
df_bar_all = (
    df_disposed
    .with_columns(
        pl.when(pl.col('created_on').dt.month() >= 7)
          .then(pl.col('created_on').dt.year())
          .otherwise(pl.col('created_on').dt.year() - 1)
          .alias('fy_jul')
    )
    .filter(pl.col('fy_jul').is_in([2021, 2022, 2023, 2024]))
    .group_by('fy_jul')
    .agg(
        pl.col('disposal_days').median().alias('median_days'),
        pl.col('disposal_days').count().alias('n'),
    )
    .sort('fy_jul')
)

bar_df = df_bar_all.to_pandas()
fy_labels = {
    2021: 'Jul 21-Jun 22', 2022: 'Jul 22-Jun 23',
    2023: 'Jul 23-Jun 24', 2024: 'Jul 24-Jun 25',
}
bar_df['label'] = bar_df['fy_jul'].map(fy_labels)
bar_df = bar_df.sort_values('fy_jul')

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(bar_df['label'], bar_df['median_days'],
              color=MAROON, width=0.52, zorder=3)

for bar, val, n in zip(bars, bar_df['median_days'], bar_df['n']):
    # value label above bar
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{val:.0f}d', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=NEAR_BLACK)
    # sample size inside bar
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
            f'n={n:,}', ha='center', va='center',
            fontsize=8, color='white')

ax.set_ylabel('Median days to disposal', fontsize=10)
ax.set_ylim(0, bar_df['median_days'].max() * 1.28)
ax.spines['left'].set_color(GREY_AXIS)
ax.spines['bottom'].set_color(GREY_AXIS)
ax.set_title(
    'Median disposal time — all disposed complaints\n'
    'Disposed complaints only, July-June fiscal years',
    fontsize=11,
)

plt.tight_layout()
fig.savefig(OUTPUT / 'disposal_time_median_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/disposal_time_median_bar.png')


---
## 4. Rural housing regression

Three OLS specifications estimated in **both levels and logs**:
- **M1**: month-year FE + office FE + 1{Online} + 1{Female}  (baseline)
- **M2**: M1 + district FE
- **M3**: M1 + block FE

M2 and M3 are separate regressions — block FE nests district FE, so they are not included jointly (collinear since districts are unions of blocks).

`office` encodes the first routing office (Collector, CM Office, Departments, Governor, Chief Secretary, SP) — 6 categories, 5 dummies, drop-first.

**Why log at all?** Disposal time is right-skewed; log-OLS gives better-behaved residuals and is conventional.  
**Why also levels?** Coefficients are directly interpretable as days — cleaner for policy communication.  
The R² comparison across specifications is the main goal: how much disposal-time variance do administrative variables capture vs what remains (attributable to complaint-level content: specificity, complexity, sentiment).


In [ ]:
# ── Rural housing disposed complaints ─────────────────────────────────────────
df_rh = (
    df_disposed
    .filter(pl.col("subcategory").is_in(RURAL_HOUSING_SUBCATS))
    .filter(pl.col("disposal_days") > 0)
    .filter(pl.col("district").is_not_null())
    .filter(pl.col("block").is_not_null())
    .filter(pl.col("mode").is_not_null())
    .filter(pl.col("petitioner_gender").is_not_null())
    .filter(pl.col("office").is_not_null())
    .filter(pl.col("fiscal_year").is_in(PRIMARY_FYS))
    .with_columns([
        pl.col("disposal_days").log1p().alias("log_disposal"),
        (pl.col("year").cast(pl.Utf8) + "-" + pl.col("month").cast(pl.Utf8)).alias("month_year"),
        # Collapse mode: Online = digital channels, Offline = in-person / paper
        pl.when(pl.col("mode").is_in([
            "Website", "Whatsapp", "Twitter", "Mobile", "Email", "Facebook"
        ])).then(pl.lit(1)).otherwise(pl.lit(0)).alias("online"),
        # Female indicator
        pl.when(pl.col("petitioner_gender").str.to_lowercase() == "female")
          .then(pl.lit(1)).otherwise(pl.lit(0)).alias("female"),
    ])
)

print(f"Rural housing disposed complaints for regression: {len(df_rh):,}")
print(df_rh.group_by("subcategory").len().sort("len", descending=True))
print("\nMode → online mapping:")
print(df_rh.group_by(["mode", "online"]).len().sort("len", descending=True))
print("\nOffice distribution:")
print(df_rh.group_by("office").len().sort("len", descending=True))


In [ ]:
# ── Convert to pandas for statsmodels ────────────────────────────────────────
reg_cols = ["log_disposal", "disposal_days", "district", "block", "office",
            "online", "female", "month_year", "fiscal_year", "subcategory"]
rh_pd = df_rh.select(reg_cols).to_pandas()

# Sanitise strings for dummy encoding (spaces/slashes/special chars → underscores)
for col in ["district", "block", "month_year", "office"]:
    rh_pd[col] = rh_pd[col].astype(str).str.strip().str.replace(r'[^\w]', '_', regex=True)

rh_pd = rh_pd.dropna(subset=["log_disposal", "disposal_days", "online", "female",
                               "district", "block", "month_year", "office"])
print(f"Regression sample: {len(rh_pd):,} rows")
print(f"Online share: {rh_pd['online'].mean():.1%}   Female share: {rh_pd['female'].mean():.1%}")
print(f"Office categories: {sorted(rh_pd['office'].unique())}")
rh_pd.head(3)


In [ ]:
import time

# ── Build design matrices once ────────────────────────────────────────────────
# Layout: [const | month-year FE | online | female | office FE | geo FE]
# online_idx and female_idx are fixed regardless of which geo FE is appended.

X_time   = pd.get_dummies(rh_pd["month_year"], drop_first=True, dtype=np.float32)
X_office = pd.get_dummies(rh_pd["office"],     drop_first=True, dtype=np.float32)
X_dist   = pd.get_dummies(rh_pd["district"],   drop_first=True, dtype=np.float32)
X_block  = pd.get_dummies(rh_pd["block"],      drop_first=True, dtype=np.float32)

scalars   = rh_pd[["online", "female"]].astype(np.float32).reset_index(drop=True)
intercept = pd.DataFrame({"const": np.ones(len(rh_pd), dtype=np.float32)})

# Indices for HC1 SEs — position is const + time dummies + online/female
online_idx = 1 + X_time.shape[1]       # intercept + month-year dummies
female_idx = online_idx + 1

X1 = pd.concat([intercept, X_time, scalars, X_office],          axis=1).values  # baseline
X2 = pd.concat([intercept, X_time, scalars, X_office, X_dist],  axis=1).values  # + district FE
X3 = pd.concat([intercept, X_time, scalars, X_office, X_block], axis=1).values  # + block FE

print(f"Office dummies: {X_office.shape[1]}  (categories: {list(X_office.columns)})")
print(f"Design matrix shapes: base={X1.shape}  dist={X2.shape}  block={X3.shape}")

# ── Fast OLS via numpy lstsq ──────────────────────────────────────────────────
def ols_numpy(X, y):
    """Returns (beta, R², adj-R², residuals) using numpy least-squares."""
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid  = y - X @ beta
    ss_res = resid @ resid
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = float(1 - ss_res / ss_tot)
    n, p = X.shape
    r2_adj = float(1 - (1 - r2) * (n - 1) / (n - p - 1))
    return beta, r2, r2_adj, resid

# ── Outcomes ──────────────────────────────────────────────────────────────────
y_log = rh_pd["log_disposal"].values.astype(np.float32)
y_lev = rh_pd["disposal_days"].values.astype(np.float32)

# ── Fit all 6 models ──────────────────────────────────────────────────────────
t0 = time.time()
results_raw = {}
for outcome_name, y in [("log", y_log), ("levels", y_lev)]:
    for spec_name, X in [("M1: baseline", X1), ("M2: + district FE", X2), ("M3: + block FE", X3)]:
        beta, r2, r2_adj, resid = ols_numpy(X, y)
        results_raw[(outcome_name, spec_name)] = {
            "beta": beta, "r2": r2, "r2_adj": r2_adj, "resid": resid, "X": X
        }
print(f"6 models solved in {time.time()-t0:.2f}s")

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for outcome_name, y in [("log(days+1)", y_log), ("days (levels)", y_lev)]:
    ok = "log" if "log" in outcome_name else "levels"
    for spec_name, X in [("M1: baseline", X1), ("M2: + district FE", X2), ("M3: + block FE", X3)]:
        r  = results_raw[(ok, spec_name)]
        b_on = float(r["beta"][online_idx])
        b_fe = float(r["beta"][female_idx])
        rows.append({
            "Model":     spec_name,
            "Outcome":   outcome_name,
            "N":         len(y),
            "R²":        round(r["r2"],    4),
            "Adj. R²":   round(r["r2_adj"],4),
            "β(online)": round(b_on, 3),
            "β(female)": round(b_fe, 3),
        })

results_df = pd.DataFrame(rows)
results_df["ΔR² vs M1"] = results_df.groupby("Outcome")["R²"].transform(
    lambda x: (x - x.iloc[0]).round(4)
)
display(results_df)

# ── HC1 SEs for online and female in M3 only ─────────────────────────────────
def hc1_se_for_indices(X, resid, idx_list):
    """HC1 sandwich SEs for specific coefficient indices only."""
    n, p = X.shape
    scale = n / (n - p)
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = (X * (resid[:, None] ** 2 * scale)).T @ X
    sandwich = XtX_inv @ meat @ XtX_inv
    return {i: float(np.sqrt(sandwich[i, i])) for i in idx_list}

t1 = time.time()
se_log_m3 = hc1_se_for_indices(X3, results_raw[("log",    "M3: + block FE")]["resid"], [online_idx, female_idx])
se_lev_m3 = hc1_se_for_indices(X3, results_raw[("levels", "M3: + block FE")]["resid"], [online_idx, female_idx])
print(f"HC1 SEs in {time.time()-t1:.2f}s")

print("\nM3 reported coefficients (HC1 SEs):")
for label, betas, ses in [
    ("log",    results_raw[("log",    "M3: + block FE")]["beta"], se_log_m3),
    ("levels", results_raw[("levels", "M3: + block FE")]["beta"], se_lev_m3),
]:
    unit = "" if label == "log" else "d"
    for name, idx in [("online", online_idx), ("female", female_idx)]:
        print(f"  [{label}] β({name}) = {betas[idx]:+.3f}{unit}  SE = {ses[idx]:.3f}{unit}")


In [ ]:
# ── Block-level coefficients from numpy M3 ────────────────────────────────────
# Block dummies start after: intercept(1) + month_year + online + female
block_start = 1 + X_time.shape[1] + 2  # = female_idx + 1

block_names_log   = X_block.columns.tolist()   # drop_first removed the reference block
block_names_level = X_block.columns.tolist()

beta_log_m3   = results_raw[("log",    "M3: + block FE")]["beta"]
beta_lev_m3   = results_raw[("levels", "M3: + block FE")]["beta"]

block_coefs = pd.DataFrame({
    "block":          block_names_log,
    "coef_log":       beta_log_m3[block_start : block_start + len(block_names_log)],
    "coef_days":      beta_lev_m3[block_start : block_start + len(block_names_level)],
})
block_coefs["implied_pct_change"] = (np.expm1(block_coefs["coef_log"]) * 100).round(1)
block_coefs = block_coefs.sort_values("coef_days", ascending=False).reset_index(drop=True)

print(f"Block coefficient range:")
print(f"  Levels: {block_coefs['coef_days'].min():.1f} to {block_coefs['coef_days'].max():.1f} days")
print(f"  Log:    {block_coefs['coef_log'].min():.3f} to {block_coefs['coef_log'].max():.3f}")
print(f"\nTop 10 slowest blocks (days above reference block):")
display(block_coefs.head(10))
print(f"\nTop 10 fastest blocks:")
display(block_coefs.tail(10))

In [ ]:
# ── Save regression results ───────────────────────────────────────────────────
reg_out = OUTPUT / "tables" / "rural_housing_regression.xlsx"

with pd.ExcelWriter(reg_out, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="model_comparison", index=False)
    block_coefs.to_excel(writer, sheet_name="block_coefficients", index=False)

    # SE table for the two reported coefficients
    se_table = pd.DataFrame([
        {"outcome": "log(days+1)", "variable": "online", "beta": float(beta_log_m3[online_idx]), "SE_HC1": se_log_m3[online_idx]},
        {"outcome": "log(days+1)", "variable": "female", "beta": float(beta_log_m3[female_idx]), "SE_HC1": se_log_m3[female_idx]},
        {"outcome": "days (levels)", "variable": "online", "beta": float(beta_lev_m3[online_idx]), "SE_HC1": se_lev_m3[online_idx]},
        {"outcome": "days (levels)", "variable": "female", "beta": float(beta_lev_m3[female_idx]), "SE_HC1": se_lev_m3[female_idx]},
    ])
    se_table.to_excel(writer, sheet_name="reported_coef_SEs", index=False)

print(f"Saved: {reg_out}")

---
## Summary of outputs

| Output | File | Notes |
|--------|------|-------|
| Petitioner frequency table | `output/tables/petitioner_frequency_distribution.xlsx` | Shares + cumulative shares |
| Repeat-filer subcategory | `output/tables/repeat_filer_subcategory_analysis.xlsx` | Same vs different subcategory |
| Disposal time histograms | `output/disposal_time_histogram.png`, `disposal_time_overlaid.png` | Disposed only, capped at 180d |
| Rural housing regression | `output/tables/rural_housing_regression.xlsx` | M1/M2/M3 comparison + block coefs |